<div style="display: flex; align-items: center;">
  <img src="../Images/Logos/DecisionIntelligence-DecisionFraming-Logo.png" width="60px" style="margin-right: 10px;">
  <span style="font-size: 1.5em; font-weight: bold;">Decision Framing - Six Thinking Hats</span>
</div>

Decision Intelligence applied in this module:

* Using decision frames to evaluate a major purchase from multiple perspectives.
* Applying Edward de Bono's Six Thinking Hats as reusable decision lenses.
* Asking GenAI to prioritize the most useful frames for a high-stakes decision.
* Combining prompt design and Microsoft.Extensions.AI to produce strict Markdown-table outputs.

-----
### Step 1 - Initialize Configuration Builder & Build the AI Extensions Responses API Orchestration

In [3]:
#r "nuget: Microsoft.Extensions.Configuration, 10.0.9"
#r "nuget: Microsoft.Extensions.Configuration.Json, 10.0.9"
#r "nuget: System.Text.Json, 10.0.9"

using Microsoft.Extensions.Configuration.Json;
using Microsoft.Extensions.Configuration;
using System.IO;
using System;

var configurationBuilder = new ConfigurationBuilder()
    .SetBasePath(Directory.GetCurrentDirectory())
    .AddJsonFile("local.settings.json", optional: true, reloadOnChange: true)
    .AddJsonFile("secrets.settings.json", optional: true, reloadOnChange: true);
var config = configurationBuilder.Build();

// IMPORTANT: You ONLY NEED either Azure OpenAI or OpenAI connection info, not both.
// Azure OpenAI Connection Info
var azureOpenAIEndpoint = config["AzureOpenAI:Endpoint"];
var azureOpenAIAPIKey = config["AzureOpenAI:APIKey"];
var azureOpenAIModelDeploymentName = config["AzureOpenAI:ModelDeploymentName"];
// OpenAI Connection Info 
var openAIAPIKey = config["OpenAI:APIKey"];
var openAIModelId = config["OpenAI:ModelId"];
// AzureOpenAI or OpenAI selection flag
var useAzureOpenAI = bool.Parse(config["AzureOpenAI:UseAzureOpenAI"] ?? "true");

Installed Packages Microsoft.Extensions.Configuration, 10.0.9 Microsoft.Extensions.Configuration.Json, 10.0.9 System.Text.Json, 10.0.9

In [5]:
// Import the Microdoft Extensions AI NuGet Packages
#r "nuget: Microsoft.Extensions.AI, 10.7.0"
#r "nuget: Microsoft.Extensions.AI.Abstractions, 10.7.0"
#r "nuget: Microsoft.Extensions.AI.OpenAI, 10.7.0"
// Import Azure & OpenAI NuGet Packages
#r "nuget: Azure.Identity, 1.21.0"
#r "nuget: OpenAI, 2.11.0"

using Azure;
using Microsoft.Extensions.AI;
using Microsoft.Extensions.Configuration;
using OpenAI;
using System.ClientModel;
using System.ComponentModel;
using System.Text.Json;


// Create the IChatClient based on the selected service
IChatClient chatClient;

// Create a new MEAI ChatClient instance
if (useAzureOpenAI)
{
    Console.WriteLine("Using Azure OpenAI Service");

    var apiKeyCredential = new ApiKeyCredential(azureOpenAIAPIKey!);

    var azureOpenAIClient = new OpenAIClient(
        apiKeyCredential,
        new OpenAIClientOptions
        {
            Endpoint = new Uri($"{azureOpenAIEndpoint!.TrimEnd('/')}/openai/v1")
        });

    #pragma warning disable OPENAI001
    var responsesClient = azureOpenAIClient.GetResponsesClient();
    chatClient = responsesClient.AsIChatClient(azureOpenAIModelDeploymentName);
}
else
{
    Console.WriteLine("Using OpenAI Service");

    var apiKeyCredential = new ApiKeyCredential(azureOpenAIAPIKey);
    var openAIClient = new OpenAIClient(apiKeyCredential);

    #pragma warning disable OPENAI001
    var responsesClient = openAIClient.GetResponsesClient();
    chatClient = responsesClient.AsIChatClient(defaultModelId: openAIModelId);
}

#pragma warning restore OPENAI001

var decisionFrameSystemPrompt = """
You are a Decision Intelligence assistant.
Help the user explore options, evaluate tradeoffs, reason through uncertainty, solve problems,
and apply systems thinking to personal, professional, strategic, and operational decisions.

Provide responses that are structured, logical, balanced, analytical, and pragmatic.
Improve the user's judgment rather than simply making the choice for them.

Output Format Instructions:
Return only a valid Markdown table.
Do not include prose before or after the table.
Do not wrap the table in triple backticks.
Do not use Markdown headings anywhere in the response.
Do not use #, ##, ###, ####, or any heading syntax inside or outside table cells.
Do not use horizontal rules of any kind.
Do not use ---, ***, or ___ anywhere in the response.
Do not use bullet lists or numbered lists inside table cells.
Use plain text column headers only.
If line breaks are needed inside a cell, use <br>.
Escape any literal pipe characters inside cell content so the table structure is not broken.
Make the table syntactically valid Markdown with exactly one header row and one separator row.
""";

async Task<string> RunDecisionFramePromptAsync(string userPrompt, ChatOptions options = null)
{
    var chatMessages = new List<ChatMessage>
    {
        new(ChatRole.System, decisionFrameSystemPrompt),
        new(ChatRole.User, userPrompt)
    };

    ChatResponse response = await chatClient.GetResponseAsync(chatMessages, options);
    return response.Text ?? string.Empty;
}

#pragma warning disable OPENAI001

var decisionFrameChatOptions = new ChatOptions
{
    RawRepresentationFactory = _ => new OpenAI.Responses.CreateResponseOptions
    {
        Model = useAzureOpenAI ? azureOpenAIModelDeploymentName : openAIModelId,
        ReasoningOptions = new OpenAI.Responses.ResponseReasoningOptions
        {
            ReasoningEffortLevel = OpenAI.Responses.ResponseReasoningEffortLevel.Medium,
            ReasoningSummaryVerbosity = OpenAI.Responses.ResponseReasoningSummaryVerbosity.Detailed
        },
        StoredOutputEnabled = false
    }
};

#pragma warning restore OPENAI001

Installed Packages Azure.Identity, 1.21.0 Microsoft.Extensions.AI, 10.7.0 Microsoft.Extensions.AI.Abstractions, 10.7.0 Microsoft.Extensions.AI.OpenAI, 10.7.0 OpenAI, 2.11.0

Using Azure OpenAI Service


-----
### Step 2 - Decision Frames (Thinking Hats) Scenario for a Major Purchase

> 📜 **_"The mere formulation of a problem is far more essential than its solution, which may be merely a matter of mathematical or experimental skill. To raise new questions, new possibilities, to regard old problems from a new angle requires creative imagination and marks real advances in science."_**
>
> -- <cite>Albert Einstein (Theoretical Physicist, Best known for developing the theory of relativity)</cite>  

The code below lists potential decision frames using the Six Thinking Hats framework for purchasing a house. Generative AI can help apply each hat as a distinct decision lens, making it easier to inspect the same choice from multiple useful perspectives.

In [6]:
// Decision Frames Prompt that lists the Six Thinking Hats approach to purchasing a house.
var decisionFramesPrompt = """
You are planning on purchasing a new home.
Use Edward de Bono's Six Thinking Hats to frame the decision-making process.

Instructions:
Create one row for each of the six thinking hats.
For each row, explain how that decision frame should guide the house-purchase decision.
Include practical questions and evidence the buyer should gather.

Output Requirements:
Return one Markdown table with these columns:
Decision Frame | Focus Area | House-Purchase Questions | Evidence To Gather | Decision Value
""";

var decisionFramesResponseText = await RunDecisionFramePromptAsync(
    decisionFramesPrompt,
    decisionFrameChatOptions);

decisionFramesResponseText.DisplayAs("text/markdown");

| Decision Frame | Focus Area | House-Purchase Questions | Evidence To Gather | Decision Value |
|---|---|---|---|---|
| White Hat: Facts and Information | Objective property, financial, and market data | What is the total purchase cost, including taxes, insurance, maintenance, closing costs, and likely renovations? How does the monthly payment compare with reliable household income? What are the property’s age, size, condition, energy costs, and recent sale prices nearby? | Mortgage preapproval, interest-rate options, inspection report, survey, title records, tax history, utility bills, insurance quotes, comparable sales, homeowners association documents, and a full cash-flow budget | Establishes a factual baseline and exposes missing information before emotion or assumptions influence the decision |
| Red Hat: Feelings and Intuition | Emotional response, comfort, identity, and unease | How do I feel when entering and leaving the property? Does the location feel safe and suitable? Am I excited about this home, or mainly afraid of missing out? Are there concerns I cannot yet explain? | Written reactions after each viewing, separate buyer preferences, feedback from trusted people, neighborhood visits at different times, and a record of emotional triggers or reservations | Recognizes important personal signals while preventing unexamined emotions from becoming the sole basis for a major financial commitment |
| Black Hat: Risks and Caution | Downside exposure, constraints, and failure scenarios | What could make this purchase unaffordable or unsuitable? What if income falls, rates rise, repairs are larger than expected, or the home takes longer to sell? Are there flood, fire, structural, legal, environmental, or neighborhood risks? | Independent inspection, flood and hazard maps, insurance availability and premiums, repair estimates, legal and title review, lending stress test, employment stability assessment, resale analysis, and contingency reserves | Prevents avoidable mistakes by identifying vulnerabilities, worst cases, and conditions that should stop or renegotiate the purchase |
| Yellow Hat: Benefits and Opportunity | Value, advantages, resilience, and future usefulness | What benefits does this home provide compared with renting or other properties? Does it support work, family, lifestyle, accessibility, or long-term plans? Could improvements increase usefulness or value? | Comparison with realistic alternatives, location and commute analysis, school or service information where relevant, renovation feasibility, projected ownership costs, and evidence of neighborhood demand | Clarifies why the purchase may be worthwhile and distinguishes durable benefits from optimistic speculation |
| Green Hat: Alternatives and Creativity | New possibilities, deal structures, and solutions | Could I buy a different property, wait, rent longer, expand the search area, or change the financing structure? Can defects be negotiated, repaired, staged, or converted into useful features? What nontraditional options reduce risk? | Alternative property shortlist, rent-versus-buy scenarios, renovation concepts and quotes, seller concessions, rate-buydown options, shared-equity or other financing terms, and flexibility analysis | Expands the choice set, reducing the risk of treating one attractive house as the only viable opportunity |
| Blue Hat: Process and Decision Control | Decision rules, sequencing, governance, and review | What are my must-have criteria, walk-away conditions, and maximum affordable price? Which checks must occur before making an offer or removing contingencies? Who provides independent challenge? When will I make the final decision? | Written decision criteria, weighted comparison matrix, affordability ceiling, due-diligence timeline, professional roles, offer and contingency checklist, scenario results, and a documented final rationale | Converts the six perspectives into a disciplined process, limits bias and urgency, and ensures the decision is revisited when evidence changes |

This can be enhanced further by asking GenAI to suggest which decision frames might be most useful for a high-stakes decision. The prompt below lists all of the decision frames, prioritizes them by expected impact and influence, and recommends which frames deserve the most attention.

This lets the AI automation help optimize the decision process while the human decision maker remains responsible for the actual decision selection and execution.

In [7]:
// Decision Frames Prompt that ranks the Six Thinking Hats approach to purchasing a house.
var decisionFramesListPrompt = """
You are planning on purchasing a new home.
Use Edward de Bono's Six Thinking Hats to frame the decision-making process.

Instructions:
List the six thinking hats in the order of expected impact and influence from most to least for a house-purchase decision.
For each thinking hat, explain why it has that ranking.
Recommend whether the buyer should focus on that hat heavily, moderately, or lightly.

Output Requirements:
Return one Markdown table with these columns:
Rank | Decision Frame | Why It Matters For A House Purchase | Impact / Influence | Recommended Focus
""";

var decisionFramesListResponseText = await RunDecisionFramePromptAsync(
    decisionFramesListPrompt,
    decisionFrameChatOptions);

decisionFramesListResponseText.DisplayAs("text/markdown");

| Rank | Decision Frame | Why It Matters For A House Purchase | Impact / Influence | Recommended Focus |
|---:|---|---|---|---|
| 1 | White Hat: Facts and Evidence | Establishes affordability, financing terms, market data, inspection findings, taxes, insurance, maintenance costs, location factors, and legal details. Reliable facts prevent decisions based on assumptions or incomplete information. | Very high: factual errors can create long-term financial stress and poor valuation decisions. | Heavily |
| 2 | Black Hat: Risks and Caution | Tests downside scenarios such as unaffordable payments, hidden defects, falling property values, unfavorable contingencies, climate exposure, and costly repairs. | Very high: the financial consequences of overlooking major risks are substantial and often difficult to reverse. | Heavily |
| 3 | Blue Hat: Process and Control | Structures the decision through criteria, budget limits, due diligence, offer strategy, professional input, deadlines, and a clear go or no-go process. | High: disciplined process reduces emotional bias, rushed decisions, and missed checks. | Heavily |
| 4 | Red Hat: Feelings and Intuition | Captures emotional fit, comfort, sense of belonging, stress, concerns about the neighborhood, and reactions that may not yet be fully rationalized. | High: emotional satisfaction affects long-term wellbeing, but feelings can also encourage overpaying or ignoring risks. | Moderately |
| 5 | Yellow Hat: Benefits and Value | Identifies potential advantages such as lifestyle improvement, stability, appreciation potential, convenience, community, rental potential, and alignment with future plans. | Moderate: benefits help compare homes and justify the purchase, but optimistic assumptions should be tested against evidence. | Moderately |
| 6 | Green Hat: Alternatives and Creativity | Encourages considering other neighborhoods, property types, renovation options, different financing structures, renting longer, or delaying the purchase. | Moderate to lower: alternatives can materially improve the outcome, but they generally influence the choice after core facts, risks, and constraints are understood. | Lightly to moderately |

-----
### Step 3 - Applying the Decision Frame to a Decision 

So far, Generative AI has helped create decision frames with the Six Thinking Hats framework, rank them from most to least impactful, and identify where to focus attention. The next step is to apply the selected decision frames directly to the house-purchase decision.

This notebook uses a single optimized prompt for that final step: the model performs the ranking and frame selection internally, then returns only the applied decision-frame approach as a Markdown table.

In [8]:
// Decision Frames Prompt that internally ranks and selects the strongest frames, then only returns the applied approach.
var optimizedDecisionFramesPrompt = """
You are planning on purchasing a new home.
Use Edward de Bono's Six Thinking Hats to frame the decision-making process.

Internal Instructions:
Internally identify an approach for each of the six thinking hats.
Internally rank the thinking hats by expected impact and influence for the house-purchase decision.
Internally select the most useful decision frames to focus on.
Do not show the internal ranking, hidden selection notes, or hidden reasoning steps.

Visible Output Instructions:
From the internally selected decision frames, create an applied approach for purchasing a house.
The response should help the buyer know what to inspect, what questions to ask, what evidence to gather, and what action to take.

Output Requirements:
Return one Markdown table with these columns:
Selected Decision Frame | Applied House-Purchase Approach | Key Questions | Evidence / Signals | Buyer Action
""";

var optimizedDecisionFramesResponseText = await RunDecisionFramePromptAsync(
    optimizedDecisionFramesPrompt,
    decisionFrameChatOptions);

optimizedDecisionFramesResponseText.DisplayAs("text/markdown");

| Selected Decision Frame | Applied House-Purchase Approach | Key Questions | Evidence / Signals | Buyer Action |
| - | - | - | - | - |
| White Hat: Facts and evidence | Establish an objective picture of the property, transaction, and ongoing ownership costs before forming a preference. | What is the verified purchase price, floor area, age, ownership history, and time on market?<br>What are the taxes, insurance, utilities, maintenance, and association fees?<br>Are permits, boundaries, disclosures, and legal title clear? | Comparable recent sales, official property records, inspection report, survey, utility bills, tax records, insurance quote, seller disclosures, and association documents. | Create a property fact sheet and verify important claims independently. Do not make an offer until material unknowns have owners, deadlines, and acceptable answers. |
| Red Hat: Emotions and intuition | Treat emotional reactions as useful signals, while separating genuine fit from excitement, fear, scarcity pressure, or attachment to appearance. | How do I feel about living here on an ordinary weekday?<br>What concerns keep recurring even when I try to dismiss them?<br>Am I responding to the home itself or to staging, competition, or fear of missing out? | Notes after visiting at different times, reactions from trusted observers, commute trial, neighborhood walk, and a written record of concerns before discussing price. | Record immediate reactions privately, then investigate recurring concerns. Pause if emotion is driving a higher offer, waived safeguards, or acceptance of an obvious compromise. |
| Black Hat: Risks and downside | Stress-test the purchase for physical, financial, legal, environmental, and lifestyle risks, including plausible adverse scenarios. | What could require a major repair soon?<br>Could rates, income, taxes, insurance, or utilities make the home unaffordable?<br>Are there flood, fire, drainage, mold, structural, zoning, noise, safety, or resale risks?<br>What happens if the home takes longer to sell later? | Independent inspection, specialist reports, flood and hazard maps, insurance availability and exclusions, repair history, permits, neighborhood crime and infrastructure data, lender appraisal, and conservative affordability scenarios. | Set non-negotiable deal-breakers, retain inspection and financing contingencies, maintain a cash reserve, and walk away when downside exceeds the household’s tolerance. |
| Yellow Hat: Benefits and value | Test the realistic upside and long-term usefulness of the home rather than relying on optimistic appreciation assumptions. | Does the location support work, schooling, health, transport, relationships, and daily routines?<br>Which features reduce future costs or improve flexibility?<br>Would the home remain valuable if prices stagnate? | Commute measurements, service and school information, planned infrastructure, rental or resale comparables, energy performance, layout adaptability, and estimated value of included improvements. | Identify benefits that matter to the buyer’s life and assign conservative value to them. Prefer durable utility and flexibility over speculative appreciation. |
| Green Hat: Alternatives and creative options | Expand the choice set and find ways to reduce risk or improve fit before treating this property as the only opportunity. | Could a different neighborhood, smaller home, renovation, new build, or continued renting meet the same goals?<br>Can price, closing date, repairs, credits, or contingencies be negotiated?<br>Can a room, layout, or energy system be adapted? | Market inventory, rejected and accepted alternatives, renovation estimates, rent-versus-buy analysis, financing products, seller constraints, and opportunity cost of the down payment. | Compare at least three viable alternatives using the same criteria. Consider a conditional offer, repair credit, phased improvements, or waiting rather than stretching to secure this home. |
| Blue Hat: Process and decision control | Run a disciplined decision process with explicit criteria, thresholds, independent checks, and a defined stop or proceed point. | What must be true for this purchase to work?<br>Which criteria are essential, desirable, or negotiable?<br>Who has authority to approve the offer?<br>What evidence would change the decision?<br>What is the maximum all-in price? | Written affordability limit, weighted scorecard, inspection and appraisal deadlines, lender preapproval, solicitor or attorney review, decision log, and independent second opinions. | Set a maximum all-in budget before bidding, complete due diligence in sequence, review evidence at a scheduled decision meeting, and proceed only if essential criteria pass and risks are funded. |